# Try SendSoon in Google Colab

Calls the same HTTP APIs as [`sendsoon/mcp`](https://github.com/sendsoon/mcp). Uses **`requests` only** (preinstalled in Colab — no `pip install`).

| MCP tool | Endpoint |
| --- | --- |
| `ip_lookup` | `GET /api/ip/lookup` |
| `markitdown_convert` | `POST /api/markitdown/convert` |
| `send_email` | `POST /api/send-test-email` |

Env (same as MCP): `SENDSOON_API_BASE_URL` (default `https://www.sendsoonai.com`), optional `SENDSOON_API_KEY`.

In [ ]:
import html
import json
import os
import uuid
from getpass import getpass

import requests

BASE = os.environ.get("SENDSOON_API_BASE_URL", "https://www.sendsoonai.com").rstrip("/")
API_KEY = os.environ.get("SENDSOON_API_KEY", "").strip() or None
recipient = input("Recipient for send_email (blank = skip): ").strip()
key_in = getpass("API Key (Enter = skip): ").strip()
if key_in:
    API_KEY = key_in

def headers(accept="application/json"):
    h = {"Accept": accept}
    if API_KEY:
        h["Authorization"] = f"Bearer {API_KEY}"
    return h

print("Base URL:", BASE)
print("API Key:", "set" if API_KEY else "anonymous trial")
print("Recipient:", recipient or "(skip send_email)")
print()

In [ ]:
# ip_lookup
r = requests.get(f"{BASE}/api/ip/lookup", params={"ip": "8.8.8.8"}, headers=headers(), timeout=30)
r.raise_for_status()
print("ip_lookup:", json.dumps(r.json(), indent=2, ensure_ascii=False))

In [ ]:
# markitdown_convert
text = "# SendSoon sample\n\nHello from Colab.\n"
r = requests.post(
    f"{BASE}/api/markitdown/convert",
    headers=headers(accept="text/markdown, application/json"),
    files={"file": ("sample.txt", text.encode())},
    timeout=60,
)
r.raise_for_status()
if "application/json" in (r.headers.get("content-type") or ""):
    print("markitdown:", r.json().get("markdown", ""))
else:
    print("markitdown:", r.text)

In [ ]:
# send_email (skipped when recipient is blank)
if not recipient:
    print("send_email: skipped (no recipient)")
else:
    body = "Configuration successful. Sent from Google Colab."
    r = requests.post(
        f"{BASE}/api/send-test-email",
        headers={**headers(), "Content-Type": "application/json", "Idempotency-Key": str(uuid.uuid4())},
        json={
            "to": recipient,
            "subject": "SendSoon Colab test",
            "htmlContent": f'<pre style="white-space:pre-wrap">{html.escape(body)}</pre>',
        },
        timeout=30,
    )
    print(f"send_email: HTTP {r.status_code}")
    print(r.text)